# V-Max BC training on Colab

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Assumes the whole local `data/` folder was already uploaded to** `MyDrive/data/` (or wherever - edit `DRIVE_DATA_ROOT` below to match). Locally, `data/shards/bc_pools/hard` and `.../easy` are symlinks into `data/train_91f`; a browser upload reads through a symlink the same way any file read does, so the real file content should have gone up, not a dangling link. **Cell 2 below checks this before anything else runs** - if it reports 0-byte files, the upload didn't dereference cleanly and needs redoing for those two folders specifically.

Runtime > Change runtime type > select a GPU (T4 is free-tier; Colab Pro gives A100/L4).

**Why Drive at all**: Colab sessions disconnect (idle timeout / max runtime). Checkpoints are written straight to Drive (via a symlink), and BC training now supports full resume - if the session dies, just re-run the training cell with the same `name_run` and it picks up from the last checkpoint instead of restarting.

In [ ]:
!nvidia-smi

## 1. Mount Drive and sanity-check the uploaded data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this if the uploaded `data/` folder landed somewhere else in your Drive.
DRIVE_DATA_ROOT = "/content/drive/MyDrive/data"

import os
for required in [
    f"{DRIVE_DATA_ROOT}/shards/bc_pools/hard",
    f"{DRIVE_DATA_ROOT}/shards/bc_pools/easy",
    f"{DRIVE_DATA_ROOT}/eval/val_sample_shards_hanam",
]:
    assert os.path.isdir(required), f"Missing {required} - check DRIVE_DATA_ROOT or re-upload."

In [ ]:
# Real files should each be a few MB. 0 bytes (or a broken os.path.islink target)
# means the upload didn't dereference that symlink and this folder needs redoing.
import os

for pool in ["hard", "easy"]:
    d = f"{DRIVE_DATA_ROOT}/shards/bc_pools/{pool}"
    names = [n for n in sorted(os.listdir(d)) if n.endswith(".tfrecord") or "tfrecord-" in n]
    sample = os.path.join(d, names[0])
    size = os.path.getsize(sample)
    print(f"{pool}: {len(names)} files, first={names[0]} size={size/1e6:.2f}MB, is_symlink={os.path.islink(sample)}")
    assert size > 1000, f"{sample} is suspiciously small ({size} bytes) - upload likely didn't dereference this symlink."

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Copy the BC pools to local disk

Repeated shuffled reads during training are much faster off local SSD than off the Drive FUSE mount - and each pool here is ~58,925 x (hard-frac) individual files, so this copy itself will take a while over FUSE (one-time cost per Colab session). `df -h` first to make sure there's room (hard+easy together are the same ~178GB as `train_91f`, since splitting into hard/easy keeps every file).

In [ ]:
!df -h /content
!mkdir -p /content/data/shards/bc_pools
!cp -r "$DRIVE_DATA_ROOT/shards/bc_pools/hard" /content/data/shards/bc_pools/hard
!cp -r "$DRIVE_DATA_ROOT/shards/bc_pools/easy" /content/data/shards/bc_pools/easy
!ls /content/data/shards/bc_pools/hard | wc -l
!ls /content/data/shards/bc_pools/easy | wc -l
!du -sh /content/data/shards/bc_pools/hard /content/data/shards/bc_pools/easy

## 4. Wire up checkpoints (Drive, persistent)

`runs/` is symlinked into Drive so checkpoints/logs survive a disconnect.

In [ ]:
DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs
!ls -la /content/as-fast-as-anyone/V-Max/runs

## 5. Train

`@23570` / `@35355` must match the `wc -l` counts printed in step 3 - edit if they differ (they shouldn't, since this is the same 178GB dataset as the local machine).

`total_timesteps=5_000_000` is roughly one pass over the combined hard+easy pool (~59k scenarios x 80 steps). Bump it up (e.g. `20_000_000`, the framework's own default scale) once you've confirmed `train/imitation_loss` in TensorBoard is still trending down at 5M and want to keep going.

**If the session disconnects mid-run**: mount Drive + re-copy the pools (steps 1-3), then re-run this cell unchanged. `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  'mixture_datasets=[{path: /content/data/shards/bc_pools/hard/hard.tfrecord@23570, weight: 0.3}, {path: /content/data/shards/bc_pools/easy/easy.tfrecord@35355, weight: 0.7}]' \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 6. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 7. After training: sweep checkpoints on the fixed held-out set

In [ ]:
!mkdir -p /content/data/eval
!cp -r "$DRIVE_DATA_ROOT/eval/val_sample_shards_hanam" /content/data/eval/val_sample_shards_hanam

%cd /content/as-fast-as-anyone/V-Max
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_bc_run1 \
  --path_dataset /content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300 \
  --waymo_dataset true --batch_size 4